# Canonical WOMD Paper Pipeline — crash-resumable

Production entrypoint for the frozen WOMD paper protocol. Large data and generated evidence live on Google Drive from the start, so a Colab disconnect does not discard completed shards or completed training runs. `scripts/run_canonical_womd_pipeline.py` remains the owner of the Stage 1–7 scientific gates.


In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys
from google.colab import auth, drive

auth.authenticate_user()
drive.mount('/content/drive')
ROOT = Path('/content/predictive-pc-fmcw')
PERSIST = Path('/content/drive/MyDrive/predictive_pc_fmcw_canonical')
DATA = PERSIST/'womd'
PERSIST.mkdir(parents=True, exist_ok=True)
DATA.mkdir(parents=True, exist_ok=True)
REPO = 'https://github.com/panagiotagrosdouli/predictive-pc-fmcw-vehicular-communications..git'
if not ROOT.exists():
    subprocess.run(['git','clone','--depth','1',REPO,str(ROOT)], check=True)
else:
    subprocess.run(['git','-C',str(ROOT),'pull','--ff-only'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',f'{ROOT}[ml,paper]','--no-build-isolation'], check=True)
from predictive_pc_fmcw.persistence import persist_completed_file_atomic, validate_completed_file

# Persist every generated stage except Stage 2, which is frozen/versioned in Git.
paper = ROOT/'artifacts/paper_final'
persistent_paper = PERSIST/'paper_final'
persistent_paper.mkdir(parents=True, exist_ok=True)
for stage in ('01_data','03_baselines','04_learning','05_heldout','06_scheduling','07_statistics','08_release'):
    live = paper/stage
    durable = persistent_paper/stage
    durable.mkdir(parents=True, exist_ok=True)
    if live.is_symlink():
        continue
    if live.exists():
        shutil.copytree(live, durable, dirs_exist_ok=True)
        shutil.rmtree(live)
    live.symlink_to(durable, target_is_directory=True)
print('Durable workspace:', PERSIST)


## Acquire the frozen WOMD shards with interruption-safe persistence

Training shards 00000–00049 of 01000 and validation shards 00000–00039 of 00150 use the sequence local `.part` → completed local file → Drive `.persisting` → size validation → atomic rename to the final name. Stale temporary files are removed on restart. Existing finals are reused only when their byte size matches the authoritative source object; Stage 1 SHA-256/provenance verification remains the scientific authority.


In [ ]:
BUCKET = 'gs://waymo_open_dataset_motion_v_1_3_1/uncompressed/scenario'
train_dir = DATA/'training'; val_dir = DATA/'validation'
train_dir.mkdir(exist_ok=True); val_dir.mkdir(exist_ok=True)
scratch = Path('/content/womd_download_tmp'); scratch.mkdir(exist_ok=True)

def remote_size(uri):
    completed = subprocess.run(['gcloud','storage','objects','describe',uri,'--format=value(size)'], check=True, capture_output=True, text=True)
    return int(completed.stdout.strip())

def fetch_split(split_name, count, total, destination):
    for index in range(count):
        name = f'{split_name}.tfrecord-{index:05d}-of-{total:05d}'
        uri = f'{BUCKET}/{split_name}/{name}'
        target = destination/name
        expected_size = remote_size(uri)
        part = scratch/(name + '.part')
        completed_local = scratch/name
        persisting = target.with_name(target.name + '.persisting')
        part.unlink(missing_ok=True)
        completed_local.unlink(missing_ok=True)
        persisting.unlink(missing_ok=True)
        existing = validate_completed_file(target, expected_size=expected_size)
        if existing.valid:
            print('resume: verified complete', name)
            continue
        target.unlink(missing_ok=True)
        subprocess.run(['gcloud','storage','cp',uri,str(part)], check=True)
        if part.stat().st_size != expected_size:
            raise RuntimeError(f'downloaded shard size mismatch: {name}')
        os.replace(part, completed_local)
        persisted = persist_completed_file_atomic(completed_local, target, expected_size=expected_size)
        completed_local.unlink(missing_ok=True)
        if not persisted.valid:
            raise RuntimeError(f'persistence validation failed: {name}: {persisted.reason}')
        print('persisted', name, persisted.size_bytes, 'bytes')

fetch_split('training', 50, 1000, train_dir)
fetch_split('validation', 40, 150, val_dir)


## Build persistent NPZ corpora only when missing

The expensive conversion is skipped after a successful build. Outputs are written to a temporary name and atomically renamed, so an interrupted conversion cannot masquerade as a valid canonical corpus. Stage 1 remains responsible for authoritative corpus hashes and leakage/provenance gates.


In [ ]:
train_npz = DATA/'womd_training_paper.npz'
val_npz = DATA/'womd_validation_paper.npz'

def build_npz(inputs, output, fixed_split=None):
    if output.is_file() and output.stat().st_size > 0:
        print('resume: NPZ already present', output)
        return
    tmp = output.with_suffix('.building.npz')
    tmp.unlink(missing_ok=True)
    cmd = [sys.executable,str(ROOT/'scripts/01_build_official_womd_samples.py'),*map(str,inputs),'--output',str(tmp),'--max-vehicles','16']
    if fixed_split:
        cmd += ['--fixed-split', fixed_split]
    subprocess.run(cmd, cwd=ROOT, check=True)
    if not tmp.is_file() or tmp.stat().st_size == 0:
        raise RuntimeError(f'NPZ build did not produce a non-empty file: {tmp}')
    os.replace(tmp, output)
    print('persisted NPZ', output)

build_npz(sorted(train_dir.glob('training.tfrecord-*')), train_npz)
build_npz(sorted(val_dir.glob('validation.tfrecord-*')), val_npz, 'official_validation')


## Stage 1 — provenance and leakage gates

This hashes every selected TFRecord shard, audits both NPZ corpora, compares the historical training fingerprint without forcing old counts, and fail-closes on scenario leakage or validation-role violations. Stage evidence is already on Drive through the durable stage links.


In [ ]:
subprocess.run([sys.executable,str(ROOT/'scripts/run_canonical_womd_pipeline.py'),'--repo-root',str(ROOT),'--data-root',str(DATA),'--train-npz',str(train_npz),'--validation-npz',str(val_npz),'--mode','stage1'],check=True)
print('Stage 1 PASS. Review historical_fingerprint.json before claiming exact historical reproduction.')


## Full canonical run — resumable Stages 3–7

Stage 2 remains frozen in Git. Stage 4 reuses only result/checkpoint pairs that pass objective, seed, dataset and checkpoint validation. The lambda choice is reused only when its development-only selection artifact validates. Re-running this cell therefore resumes verified work and recomputes invalid partial work.


In [ ]:
LAMBDA_LINK = 0.2
LAMBDA_OUTAGE = 0.1
RATIONALE = 'Selected from the declared development-only sweep; frozen before Stage 5.'
validation_pattern = str(val_dir/'validation.tfrecord-*')
subprocess.run([sys.executable,str(ROOT/'scripts/run_canonical_womd_pipeline.py'),'--repo-root',str(ROOT),'--data-root',str(DATA),'--train-npz',str(train_npz),'--validation-npz',str(val_npz),'--validation-glob',validation_pattern,'--lambda-link',str(LAMBDA_LINK),'--lambda-outage',str(LAMBDA_OUTAGE),'--selection-rationale',RATIONALE,'--mode','full'],check=True)
print('Stages 1–7 complete; durable evidence is under', PERSIST/'paper_final')
